# Phân loại ảnh X-quang y tế: ResNet-50 vs ConvNeXt-Tiny

**Nhóm 11** – Lê Thị Khánh Linh · Nguyễn Huyền Thương · Trần Bùi Hà Giang

Pipeline:
1. Cấu hình & kiểm tra môi trường
2. Load dữ liệu + tính class weights
3. Huấn luyện ResNet-50
4. Huấn luyện ConvNeXt-Tiny
5. Đánh giá & so sánh hai mô hình
6. Vẽ biểu đồ tổng hợp

## 0. Cài đặt thư viện

In [1]:
# Chỉ chạy lần đầu nếu chưa cài
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm pandas Pillow

## 1. Import & Cấu hình

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

from src.dataset  import build_dataloaders
from src.models   import build_model
from src.train    import train_model
from src.evaluate import evaluate_model, compare_models
from src.utils    import plot_all, plot_model_comparison

# ── Cấu hình chung ───────────────────────────────────────────────────────────
CONFIG = {
    "data_dir"      : "./data",     # Thư mục chứa train/ val/ test/
    "output_dir"    : "./outputs",  # Lưu model checkpoint + biểu đồ
    "img_size"      : 224,
    "epochs"        : 20,
    "lr"            : 1e-4,
    "weight_decay"  : 1e-4,
    "dropout"       : 0.3,
    "drop_path_rate": 0.1,          # chỉ dùng cho ConvNeXt
    "patience"      : 7,            # early stopping
    "num_workers"   : 0,            # giảm xuống 0 nếu lỗi trên Windows
    "batch_resnet"  : 16,
    "batch_convnext": 16,           # ConvNeXt tốn VRAM hơn
}

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Config : {CONFIG}")

Device : cuda
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM   : 8.6 GB
Config : {'data_dir': './data', 'output_dir': './outputs', 'img_size': 224, 'epochs': 20, 'lr': 0.0001, 'weight_decay': 0.0001, 'dropout': 0.3, 'drop_path_rate': 0.1, 'patience': 7, 'num_workers': 2, 'batch_resnet': 16, 'batch_convnext': 16}


## 2. Load dữ liệu

In [3]:
# ── ResNet DataLoaders ────────────────────────────────────────────────────────
train_loader_r, val_loader_r, test_loader_r, class_names, class_weights = build_dataloaders(
    data_dir         = CONFIG["data_dir"],
    img_size         = CONFIG["img_size"],
    batch_size_train = CONFIG["batch_resnet"],
    batch_size_eval  = CONFIG["batch_resnet"],
    num_workers      = CONFIG["num_workers"],
)

# ── ConvNeXt DataLoaders (batch nhỏ hơn) ─────────────────────────────────────
train_loader_c, val_loader_c, test_loader_c, _, _ = build_dataloaders(
    data_dir         = CONFIG["data_dir"],
    img_size         = CONFIG["img_size"],
    batch_size_train = CONFIG["batch_convnext"],
    batch_size_eval  = CONFIG["batch_convnext"],
    num_workers      = CONFIG["num_workers"],
)

print(f"\nSố class : {len(class_names)}")
print(f"Classes  : {class_names}")
print(f"Weights  : {class_weights.tolist()}")

[Dataset] Class distribution : {0: 7263, 1: 4674, 2: 8513}
[Dataset] Class weights       : [0.9385469555854797, 1.4584224224090576, 0.800736129283905]
[Dataset] Train: 20450 | Val: 2534 | Test: 2569
[Dataset] Classes: ['normal', 'pneumonia', 'tuberculosis']
[Dataset] Class distribution : {0: 7263, 1: 4674, 2: 8513}
[Dataset] Class weights       : [0.9385469555854797, 1.4584224224090576, 0.800736129283905]
[Dataset] Train: 20450 | Val: 2534 | Test: 2569
[Dataset] Classes: ['normal', 'pneumonia', 'tuberculosis']

Số class : 3
Classes  : ['normal', 'pneumonia', 'tuberculosis']
Weights  : [0.9385469555854797, 1.4584224224090576, 0.800736129283905]


### Kiểm tra một batch mẫu

In [4]:
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img * STD + MEAN, 0, 1)

images, labels = next(iter(train_loader_r))
n_show = min(8, len(images))

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for i, ax in enumerate(axes):
    ax.imshow(denormalize(images[i]))
    ax.set_title(class_names[labels[i].item()], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample batch từ train set", fontsize=12)
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

## 3. Huấn luyện ResNet-50

In [ ]:
model_resnet = build_model(
    model_name = "resnet50",
    n_classes  = len(class_names),
    dropout    = CONFIG["dropout"],
)

history_resnet = train_model(
    model         = model_resnet,
    train_loader  = train_loader_r,
    val_loader    = val_loader_r,
    class_weights = class_weights,
    device        = device,
    epochs        = CONFIG["epochs"],
    lr            = CONFIG["lr"],
    weight_decay  = CONFIG["weight_decay"],
    patience      = CONFIG["patience"],
    output_dir    = CONFIG["output_dir"],
    model_name    = "resnet50",
)

e:\24022314\Năm 2\Kì 2 năm 2\Học sâu\medical-xray-vit-vs-cnn\src\train.py:237: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


[Model] ResNet-50 (tự cài đặt) | n_classes=3
[Model] Tổng params: 23,514,179 | Trainable: 23,514,179

  Bắt đầu training: RESNET50
  Epochs=20 | LR=0.0001 | Device=cuda



KeyboardInterrupt: 

### Training curves – ResNet-50

In [ ]:
from src.utils import plot_training_curves
plot_training_curves(history_resnet, model_name="resnet50", output_dir=CONFIG["output_dir"])
plt.show()

## 4. Huấn luyện ConvNeXt-Tiny

In [ ]:
model_convnext = build_model(
    model_name     = "convnext",
    n_classes      = len(class_names),
    dropout        = CONFIG["dropout"],
    drop_path_rate = CONFIG["drop_path_rate"],
)

history_convnext = train_model(
    model         = model_convnext,
    train_loader  = train_loader_c,
    val_loader    = val_loader_c,
    class_weights = class_weights,
    device        = device,
    epochs        = CONFIG["epochs"],
    lr            = CONFIG["lr"],
    weight_decay  = CONFIG["weight_decay"],
    patience      = CONFIG["patience"],
    output_dir    = CONFIG["output_dir"],
    model_name    = "convnext",
)

### Training curves – ConvNeXt-Tiny

In [ ]:
plot_training_curves(history_convnext, model_name="convnext", output_dir=CONFIG["output_dir"])
plt.show()

## 5. Đánh giá trên tập Test

In [ ]:
metrics_resnet = evaluate_model(
    model       = model_resnet,
    test_loader = test_loader_r,
    class_names = class_names,
    device      = device,
    output_dir  = CONFIG["output_dir"],
    model_name  = "resnet50",
)

In [ ]:
metrics_convnext = evaluate_model(
    model       = model_convnext,
    test_loader = test_loader_c,
    class_names = class_names,
    device      = device,
    output_dir  = CONFIG["output_dir"],
    model_name  = "convnext",
)

## 6. Biểu đồ & So sánh

In [ ]:
from src.utils import plot_confusion_matrix, plot_roc_curve

# ── Confusion Matrix ──────────────────────────────────────────────────────────
plot_confusion_matrix(
    metrics_resnet["y_true"], metrics_resnet["y_pred"],
    class_names, "resnet50", CONFIG["output_dir"],
)
plt.show()

plot_confusion_matrix(
    metrics_convnext["y_true"], metrics_convnext["y_pred"],
    class_names, "convnext", CONFIG["output_dir"],
)
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
plot_roc_curve(
    metrics_resnet["y_true"], metrics_resnet["y_proba"],
    class_names, "resnet50", CONFIG["output_dir"],
)
plt.show()

plot_roc_curve(
    metrics_convnext["y_true"], metrics_convnext["y_proba"],
    class_names, "convnext", CONFIG["output_dir"],
)
plt.show()

In [ ]:
# ── Bảng & biểu đồ so sánh ───────────────────────────────────────────────────
compare_models(metrics_resnet, metrics_convnext)
plot_model_comparison(metrics_resnet, metrics_convnext, output_dir=CONFIG["output_dir"])
plt.show()

## 7. Tổng kết

Tất cả checkpoint (`.pth`) và biểu đồ (`.png`) đã được lưu vào thư mục `outputs/`.

| File | Nội dung |
|---|---|
| `resnet50_best.pth` | Best checkpoint ResNet-50 |
| `convnext_best.pth` | Best checkpoint ConvNeXt-Tiny |
| `*_training_curves.png` | Loss & F1 theo epoch |
| `*_confusion_matrix.png` | Confusion Matrix (raw + normalized) |
| `*_roc_curve.png` | ROC Curve per-class + macro |
| `model_comparison.png` | Bar chart so sánh 2 model |